# Experiment №9
## Tangential movement over very short distances (small LR) yields almost no increase in the embedding norm and cannot overcome the random norm at initialization.

- Left tower: User embedding layer (nn.Embedding)
- Right tower: Item embedding layer (nn.Embedding)
- Cosine-based loss is used: InfoNCE
- SGD without momentum and without regularization
  
I measure orthogonality of updates per each epoch. Then I report popularity bias for item embeddings (correlation with item frequency).

## 1) Setup and configuration

- Dataclasses for model and training configs.
- You can change `cfg` and `train_cfg` to explore variants (e.g., embedding size).

In [1]:
from __future__ import annotations

# Core imports
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple, Sequence

import numpy as np
import pandas as pd
from tqdm import tqdm
import plotly.express as px
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader, Dataset
from info_nce import InfoNCE

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device: str = 'cuda' if torch.cuda.is_available() else 'cpu'

# Configuration
@dataclass
class ModelConfig:
    emb_size: int = 100

@dataclass
class TrainConfig:
    # general params:
    data_fraction: float = 1.0 # use first X fraction of ratings rows
    epochs: int = 50
    batch_size: int = 256
    lr: float = 0.1            # SGD learning rate

cfg = ModelConfig()
train_cfg = TrainConfig()

## 2) Data (MovieLens 32M) — brief EDA

Permalink: https://grouplens.org/datasets/movielens/32m/

I load ratings and movies, then report: number of users, items, interactions per user.


In [2]:
# Load MovieLens 32M and build sequences (ultra-brief EDA)
DATA_DIR = "../ml-32m"
ratings = pd.read_csv(f"{DATA_DIR}/ratings.csv")  # userId,movieId,rating,timestamp
# take only first fraction of rows for faster experiments
if train_cfg.data_fraction < 1:
    ratings = ratings[:int(len(ratings) * train_cfg.data_fraction)]

# EDA
num_users = ratings.userId.nunique()
num_items = ratings.movieId.nunique()
print("Users:", num_users, ", items:", num_items)

ratings["user_id"] = ratings["userId"].astype('str')
ratings["item_id"] = ratings["movieId"].astype('str')

ratings = ratings[["user_id", "item_id"]]

ratings

Users: 200948 , items: 84432


,user_id,item_id
0,1,17
1,1,25
2,1,29
3,1,30
4,1,32
...,...,...
32000199,200948,79702
32000200,200948,79796
32000201,200948,80350
32000202,200948,80463


## 3) Dataloader

In [3]:
user_encoder = LabelEncoder().fit(ratings["user_id"])
item_encoder = LabelEncoder().fit(ratings["item_id"])

In [4]:
class TwoTowerDataset(Dataset):

    def __init__(self,
                 positive_interactions_dataframe,
                 user_encoder, 
                 item_encoder
                ):
        self.positive_interactions_dataframe = positive_interactions_dataframe.copy()
        
        self.positive_interactions_dataframe["encoded_user_id"] = user_encoder.transform(self.positive_interactions_dataframe.user_id)
        self.positive_interactions_dataframe["encoded_item_id"] = item_encoder.transform(self.positive_interactions_dataframe.item_id)
        
        self.user2features = {}
        self.item2features = {}
        self.interactions_features_list = []

        for row in tqdm(self.positive_interactions_dataframe.itertuples()):
            self.interactions_features_list.append(
                (
                    (row.encoded_user_id,),
                    (row.encoded_item_id,)
                )
            )
            if row.user_id not in self.user2features:
                self.user2features[row.user_id] = (row.encoded_user_id,)

            if row.item_id not in self.item2features:
                self.item2features[row.item_id] = (row.encoded_item_id,)

        self.start = 0
        self.end = len(self.interactions_features_list)
    
    def __len__(self):
        return len(self.interactions_features_list)
    
    def __getitem__(self, i):
        return self.interactions_features_list[i]


train_dataset = TwoTowerDataset(ratings, user_encoder, item_encoder)

shuffle = True
num_workers = 4
batch_size = train_cfg.batch_size

def collate_fn(batch):
    users_features, items_features = zip(*batch)

    return (
        torch.IntTensor(np.array(users_features)),
        torch.IntTensor(np.array(items_features)),
    )

# Reproducibility
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(0)

train_loader = DataLoader(
    train_dataset,
    shuffle=shuffle,
    batch_size=batch_size,
    collate_fn=collate_fn,
    pin_memory=True,
    worker_init_fn=seed_worker,
    num_workers=num_workers,
    generator=g
)

32000204it [00:36, 881057.29it/s] 


In [5]:
print(f'Batch shape: \n')

some_batch = next(iter(train_loader))

some_batch[0].shape, some_batch[1].shape

Batch shape: 



(torch.Size([256, 1]), torch.Size([256, 1]))

## 4) Model (Two-Tower)

In [6]:
class TwoTowerModel(nn.Module):
    def __init__(self, user_embedding_sizes, item_embedding_sizes, device):
        super(TwoTowerModel, self).__init__()
        self.device = device

        self.user_embeds = nn.Embedding(user_embedding_sizes[0], user_embedding_sizes[1]).double()
        self.item_embeds = nn.Embedding(item_embedding_sizes[0], item_embedding_sizes[1]).double()

    def get_user_embeddings(self, user_features):
        user_embeddings = self.user_embeds(user_features[:, 0])
        return user_embeddings

    def get_item_embeddings(self, item_features):
        item_embeddings = self.item_embeds(item_features[:, 0])
        return item_embeddings

    def forward(self, user_features, item_features):
        user_embs = self.get_user_embeddings(user_features.to(self.device))
        item_embs = self.get_item_embeddings(item_features.to(self.device))

        return user_embs, item_embs

torch.cuda.empty_cache()

## 5) Training and visualization

In [7]:
def report_embedding_shift_orthogonality(
    old_users_embeddings,
    old_items_embeddings,
    new_users_embeddings,
    new_items_embeddings,
):
    old_users_embeddings = old_users_embeddings.detach().cpu().numpy()
    old_items_embeddings = old_items_embeddings.detach().cpu().numpy()
    new_users_embeddings = new_users_embeddings.detach().cpu().numpy()
    new_items_embeddings = new_items_embeddings.detach().cpu().numpy()

    def which_share_of_objects_moved_orthogonally(old, new):
        cnt = 0
        for i in range(len(old)):
            position_delta = new[i] - old[i]
            if math.isclose(float(position_delta @ old[i]), 0.0, abs_tol=1e-7):
                cnt += 1
        return cnt / len(old)

    frac_users = which_share_of_objects_moved_orthogonally(old_users_embeddings, new_users_embeddings)
    frac_items = which_share_of_objects_moved_orthogonally(old_items_embeddings, new_items_embeddings)

    print(f"Fraction of strictly orthogonal shifts (users): {frac_users:.6f}")
    print(f"Fraction of strictly orthogonal shifts (items): {frac_items:.6f}")

In [8]:
user_embedding_sizes = [len(user_encoder.classes_), cfg.emb_size]
item_embedding_sizes = [len(item_encoder.classes_), cfg.emb_size]

model = TwoTowerModel(user_embedding_sizes, item_embedding_sizes, device).to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=train_cfg.lr, momentum=0)

infonceloss = InfoNCE(temperature=0.1)

users_ids, users_features = zip(*train_dataset.user2features.items())
items_ids, items_features = zip(*train_dataset.item2features.items())

def train_loop(model, optimizer, train_loader, n_epochs=train_cfg.epochs):
    loss_history = list()
    for epoch in range(n_epochs):
        embeds_already_visualized_this_epoch = False
        
        for batch in tqdm(train_loader, desc=f'Epoch {epoch}'):
            
            user_embs, item_embs = model(*batch)
            loss = infonceloss(user_embs, item_embs)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not embeds_already_visualized_this_epoch:
                
                new_user_embs, new_item_embs = model(*batch)
    
                report_embedding_shift_orthogonality(
                    user_embs,
                    item_embs,
                    new_user_embs,
                    new_item_embs
                )
                
                embeds_already_visualized_this_epoch = True
    return

model.train()
train_loop(model, optimizer, train_loader)

Epoch 0:   0%|          | 1/125001 [00:02<70:03:10,  2.02s/it]

Fraction of strictly orthogonal shifts (users): 1.000000
Fraction of strictly orthogonal shifts (items): 1.000000


Epoch 1:   0%|          | 78/125001 [00:01<38:02, 54.73it/s]  

Fraction of strictly orthogonal shifts (users): 1.000000
Fraction of strictly orthogonal shifts (items): 1.000000


Epoch 2:   0%|          | 81/125001 [00:01<36:37, 56.85it/s]  

...

Epoch 48:   0%|          | 79/125001 [00:02<38:16, 54.40it/s]  

Fraction of strictly orthogonal shifts (users): 1.000000
Fraction of strictly orthogonal shifts (items): 1.000000


Epoch 49:   0%|          | 85/125001 [00:02<35:17, 58.99it/s]  

Fraction of strictly orthogonal shifts (users): 1.000000
Fraction of strictly orthogonal shifts (items): 1.000000


Epoch 49: 100%|██████████| 125001/125001 [02:49<00:00, 735.95it/s]


## Popularity bias

Correlation between item embedding norms and item popularity (frequency in training):

In [9]:
# Collect final item embeddings and compute popularity correlation
with torch.no_grad():
    num_items = len(item_encoder.classes_)
    all_item_feats = torch.arange(num_items, dtype=torch.long, device=device).unsqueeze(1)
    item_emb_matrix = model.get_item_embeddings(all_item_feats)

item_norms = torch.norm(item_emb_matrix, dim=1).cpu().numpy()

# Popularity from encoded training data
encoded_item_ids = item_encoder.transform(ratings["item_id"].to_numpy())
pop_arr = np.bincount(encoded_item_ids, minlength=num_items)

# Pearson correlation
corr = np.corrcoef(item_norms, pop_arr)[0, 1]
print("Correlation(norm(item), popularity):", float(corr))

Correlation(norm(item), popularity): 0.005487232220022169


## Popularity entanglement (permutation baseline)

I test whether the correlation between item embedding norms and item popularity is above what would be expected by chance.

- Null (H0): the observed correlation arises from random alignment between norms and popularity.
- Test: compute empirical correlation r_emp; then permute popularity counts K times to build a null distribution of r_perm. Report p-value and z-score.

Interpretation:
- Low p-value (e.g., < 0.01) and high z-score → popularity entanglement is significant (bad: embeddings encode popularity magnitude).
- High p-value → correlation could be random (less concerning).

In [10]:
# Permutation baseline for popularity entanglement
# Build null by permuting popularity counts K times, compute z-score, p-value
K = 200
r_perm = np.empty(K, dtype=float)
for k in range(K):
    perm = np.random.permutation(pop_arr)
    r_perm[k] = np.corrcoef(item_norms, perm)[0, 1]

mu = float(r_perm.mean())
sigma = float(r_perm.std(ddof=1) + 1e-12)
r_emp = float(corr)
z = (r_emp - mu) / sigma
from math import erf, sqrt
p_value = 2 * (1 - 0.5 * (1 + erf(abs(z) / sqrt(2))))

print(f"Empirical corr r_emp = {r_emp:.4f}")
print(f"Permutation null: mean={mu:.4f}, std={sigma:.4f}, z={z:.2f}, p-value={p_value:.3g}")

if p_value < 0.01 and r_emp > 0:
    print("Verdict: Significant popularity entanglement (bad: norms encode popularity magnitude).")
elif p_value < 0.05 and r_emp > 0:
    print("Verdict: Likely entanglement (watch out for bias).")
else:
    print("Verdict: No strong evidence beyond chance (less concerning).")

Empirical corr r_emp = 0.0055
Permutation null: mean=0.0001, std=0.0034, z=1.60, p-value=0.11
Verdict: No strong evidence beyond chance (less concerning).


## Top-1% popular share in top-100 recommendations

In [11]:
# --- Top-1% popular share in top-100 (final evaluation only) ---

# Popularity from training data (encoded ids)
num_items = len(item_encoder.classes_)
num_users = len(user_encoder.classes_)
encoded_item_ids = item_encoder.transform(ratings["item_id"].to_numpy())
pop_arr = np.bincount(encoded_item_ids, minlength=num_items)

top_1pct_k = max(1, int(math.ceil(0.01 * num_items)))
top_k_reco = min(100, num_items)
top_pop_idx = np.argpartition(pop_arr, -top_1pct_k)[-top_1pct_k:]

top_pop_mask = torch.zeros(num_items, dtype=torch.bool, device=device)
top_pop_mask[top_pop_idx] = True

rec_batch_size = 256  # adjust if needed for memory

was_training = model.training
model.eval()

cos_shares = []
dot_shares = []

with torch.no_grad():
    all_item_feats = torch.arange(num_items, dtype=torch.long, device=device).unsqueeze(1)
    item_embs = model.get_item_embeddings(all_item_feats).float()
    item_embs_norm = F.normalize(item_embs, dim=1)

    for start in range(0, num_users, rec_batch_size):
        end = min(start + rec_batch_size, num_users)
        user_feats = torch.arange(start, end, dtype=torch.long, device=device).unsqueeze(1)
        user_embs = model.get_user_embeddings(user_feats).float()

        # dot product
        scores_dot = user_embs @ item_embs.t()
        top_dot = torch.topk(scores_dot, k=top_k_reco, dim=1).indices
        dot_share = top_pop_mask[top_dot].float().mean(dim=1)
        dot_shares.append(dot_share.cpu())

        # cosine similarity
        user_embs_norm = F.normalize(user_embs, dim=1)
        scores_cos = user_embs_norm @ item_embs_norm.t()
        top_cos = torch.topk(scores_cos, k=top_k_reco, dim=1).indices
        cos_share = top_pop_mask[top_cos].float().mean(dim=1)
        cos_shares.append(cos_share.cpu())

if was_training:
    model.train()

cos_share_all = torch.cat(cos_shares).numpy()
dot_share_all = torch.cat(dot_shares).numpy()

print(f"Top-1% in top-100 (cosine): mean={cos_share_all.mean():.4f}, median={np.median(cos_share_all):.4f}")
print(f"Top-1% in top-100 (dot):    mean={dot_share_all.mean():.4f}, median={np.median(dot_share_all):.4f}")

Top-1% in top-100 (cosine): mean=0.0068, median=0.0000
Top-1% in top-100 (dot):    mean=0.0072, median=0.0000
